# **TP géoloc addresses**

## Échauffement : construire une URL avec des paramètres

In [133]:
def construire_url(num,typ,nom):
    return f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom},Paris&limit=1"

construire_url(3,"rue","tourelles")

'https://api-adresse.data.gouv.fr/search/?q=3+rue+tourelles,Paris&limit=1'

## Échauffement : sélectionner des lignes avec ```iloc``` et ```loc```

In [134]:
import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]
sub

,number,name
1,18,BERNARDINS
3,12,Rivoli


```sub``` garde les lignes du DataFrame qui ont number > 10.

In [135]:
df.iloc[:2]

,number,name
0,3,tourelles
1,18,BERNARDINS


In [136]:
df.loc[:2]

,number,name
0,3,tourelles
1,18,BERNARDINS
2,7,Bouvart


```loc``` inclut la borne supérieure, pas ```ìloc```

In [137]:
sub.iloc[:2]

,number,name
1,18,BERNARDINS
3,12,Rivoli


In [138]:
sub.loc[:2]

,number,name
1,18,BERNARDINS


```iloc``` regarde le positionnement de la ligne indépendamment de l'index, alors que ```loc```regarde l'index.

In [139]:
sub.iloc[0]

number            18
name      BERNARDINS
Name: 1, dtype: object

```sub.iloc[0]``` renvoie la première ligne de sub, et ```sub.loc[0]``` renvoie une erreur car voudrait renvoyer la ligne indexée 0 mais elle n'est pas dans sub.

In [140]:
sub.iloc[0]["name"]

'BERNARDINS'

In [141]:
sub.loc[1]["name"]

'BERNARDINS'

## Géolocalisation

In [142]:
import pandas as pd

addresses = pd.read_csv('data/addresses.csv')
addresses.head(4)

,number,type,name,city
0,5,rue,RIQUET,Paris
1,141,boulevard,ITALIE,Paris
2,234,rue,FAUBOURG ST-HONORÉ,Paris
3,62,rue,MICHEL BIZOT,Paris


In [143]:
addresses.head(20).to_csv('data/addresses-small.csv',index=False)

### Une par une

Les 2 URL renvoient des dictionnaires JSON. Avec le ```&limit=1``` on se limite à 1 adresse (la plus proche), et sans on a toutes les adresses proches.

In [144]:
import requests
import io

In [145]:
#%%timeit

def localize_one(num,typ,nom):
    url = construire_url(num,typ,nom)
    response = requests.get(url)
    if 200 <= response.status_code < 300:
        return response.json()
    else:
        return None

localize_one(18,"rue","BERNARDINS")

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.350728, 48.850345]},
   'properties': {'label': '18 Rue des Bernardins 75005 Paris',
    'score': 0.7544495104895104,
    'housenumber': '18',
    'id': '75105_0896_00018',
    'banId': 'acbebc10-eac0-475f-9466-e7dbe9446b02',
    'name': '18 Rue des Bernardins',
    'postcode': '75005',
    'citycode': '75105',
    'x': 652355.3,
    'y': 6861340.65,
    'city': 'Paris',
    'district': 'Paris 5e Arrondissement',
    'context': '75, Paris, Île-de-France',
    'type': 'housenumber',
    'importance': 0.68356,
    'depcode': '75',
    'street': 'Rue des Bernardins',
    '_type': 'address'}}],
 'query': '18 rue BERNARDINS,Paris'}

Pour faire ça 30.000 fois il faudrait environ 160 * 10^-3 * 20000 = 3200 s = 53min20s.

### En un seul coup

In [146]:
def mass_post(filename, col_number, col_type, col_name, col_city):
    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None
    result = pd.read_csv(io.StringIO(response.text))
    return result

mass_post("data/addresses-small.csv", "number", "type", "name", "city")

,number,type,name,city,longitude,latitude,result_score,result_score_next,result_label,result_type,...,result_name,result_street,result_postcode,result_city,result_context,result_citycode,result_oldcitycode,result_oldcity,result_district,result_status
0,5,rue,RIQUET,Paris,2.376448,48.888134,0.978753,0.869824,5 Rue Riquet 75019 Paris,housenumber,...,5 Rue Riquet,Rue Riquet,75019,Paris,"75, Paris, Île-de-France",75119,NaN,NaN,Paris 19e Arrondissement,ok
1,141,boulevard,ITALIE,Paris,-1.418872,46.669103,0.570112,0.554566,Boulevard d'Italie 85000 La Roche-sur-Yon,street,...,Boulevard d'Italie,Boulevard d'Italie,85000,La Roche-sur-Yon,"85, Vendée, Pays de la Loire",85191,NaN,NaN,NaN,ok
2,234,rue,FAUBOURG ST-HONORÉ,Paris,2.302766,48.876160,0.784806,0.519872,234 Rue du Faubourg Saint-Honoré 75008 Paris,housenumber,...,234 Rue du Faubourg Saint-Honoré,Rue du Faubourg Saint-Honoré,75008,Paris,"75, Paris, Île-de-France",75108,NaN,NaN,Paris 8e Arrondissement,ok
3,62,rue,MICHEL BIZOT,Paris,2.402040,48.835423,0.571099,NaN,62 Avenue du Général Michel Bizot 75012 Paris,housenumber,...,62 Avenue du Général Michel Bizot,Avenue du Général Michel Bizot,75012,Paris,"75, Paris, Île-de-France",75112,NaN,NaN,Paris 12e Arrondissement,ok
4,6,rue,OURS,Paris,2.351775,48.863311,0.773272,0.580658,6 Rue aux Ours 75003 Paris,housenumber,...,6 Rue aux Ours,Rue aux Ours,75003,Paris,"75, Paris, Île-de-France",75103,NaN,NaN,Paris 3e Arrondissement,ok
5,30,rue,PETIT-MUSC,Paris,2.364722,48.852945,0.858322,NaN,30 Rue du Petit Musc 75004 Paris,housenumber,...,30 Rue du Petit Musc,Rue du Petit Musc,75004,Paris,"75, Paris, Île-de-France",75104,NaN,NaN,Paris 4e Arrondissement,ok
6,3,impasse,MASSONET,Paris,2.351580,48.895498,0.932958,NaN,3 Impasse Massonnet 75018 Paris,housenumber,...,3 Impasse Massonnet,Impasse Massonnet,75018,Paris,"75, Paris, Île-de-France",75118,NaN,NaN,Paris 18e Arrondissement,ok
7,56,rue,PANOYAUX,Paris,2.388347,48.867023,0.828117,0.515583,56 Rue des Panoyaux 75020 Paris,housenumber,...,56 Rue des Panoyaux,Rue des Panoyaux,75020,Paris,"75, Paris, Île-de-France",75120,NaN,NaN,Paris 20e Arrondissement,ok
8,16,rue,THÉRÈSE,Paris,2.335212,48.866374,0.962650,0.440947,16 Rue Thérèse 75001 Paris,housenumber,...,16 Rue Thérèse,Rue Thérèse,75001,Paris,"75, Paris, Île-de-France",75101,NaN,NaN,Paris 1er Arrondissement,ok
9,11,rue,JEAN ROBERT,Paris,2.358170,48.889520,0.973431,0.659050,11 Rue Jean Robert 75018 Paris,housenumber,...,11 Rue Jean Robert,Rue Jean Robert,75018,Paris,"75, Paris, Île-de-France",75118,NaN,NaN,Paris 18e Arrondissement,ok


In [149]:
def localize_many(filename, col_number, col_type, col_name, col_city):
    adresses = pd.read_csv(filename)
    mass = mass_post(filename, col_number, col_type, col_name, col_city)
    mass2 = mass[["latitude","longitude","result_city","result_type"]]
    result = pd.merge(adresses,mass2,left_index=True,right_index=True)
    return result

addresses_small = localize_many("data/addresses-small.csv", "number", "type", "name", "city")
addresses_small

,number,type,name,city,latitude,longitude,result_city,result_type
0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
1,141,boulevard,ITALIE,Paris,46.669103,-1.418872,La Roche-sur-Yon,street
2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
5,30,rue,PETIT-MUSC,Paris,48.852945,2.364722,Paris,housenumber
6,3,impasse,MASSONET,Paris,48.895498,2.351580,Paris,housenumber
7,56,rue,PANOYAUX,Paris,48.867023,2.388347,Paris,housenumber
8,16,rue,THÉRÈSE,Paris,48.866374,2.335212,Paris,housenumber
9,11,rue,JEAN ROBERT,Paris,48.889520,2.358170,Paris,housenumber


In [162]:
addresses_small["result_city"].fillna("")
addresses_small[(addresses_small["result_city"] != "Paris") | (addresses_small["result_type"] != "housenumber")]

,number,type,name,city,latitude,longitude,result_city,result_type
1,141,boulevard,ITALIE,Paris,46.669103,-1.418872,La Roche-sur-Yon,street


In [164]:
addresses_small = addresses_small[(addresses_small["result_city"]=="Paris")&(addresses_small["result_type"]=="housenumber")]
addresses_small

,number,type,name,city,latitude,longitude,result_city,result_type
0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
5,30,rue,PETIT-MUSC,Paris,48.852945,2.364722,Paris,housenumber
6,3,impasse,MASSONET,Paris,48.895498,2.351580,Paris,housenumber
7,56,rue,PANOYAUX,Paris,48.867023,2.388347,Paris,housenumber
8,16,rue,THÉRÈSE,Paris,48.866374,2.335212,Paris,housenumber
9,11,rue,JEAN ROBERT,Paris,48.889520,2.358170,Paris,housenumber
10,22,rue,FAUBOURG POISSONNIÈRE,Paris,48.872540,2.348002,Paris,housenumber


In [ ]:
addresses = localize_many("data/addresses.csv", "number", "type", "name", "city")

In [168]:
addresses = addresses[(addresses["result_city"]=="Paris")&(addresses["result_type"]=="housenumber")]
len(addresses)

20678

In [171]:
addresses.to_csv("data/addresses-geoloc.csv",index=False)

## Afficher sur une carte

In [175]:
import folium

CENTER = 48.856542, 2.347614

### Le fond de carte

In [176]:
addresses = pd.read_csv("data/addresses-geoloc.csv")
addresses_small = addresses.iloc[:20]

In [181]:
def paris_map():
    m = folium.Map(location=CENTER)


folium.Map(location=CENTER)